# 🧭 트레이딩 에이전트 — Colab 퀵스타트

파이썬을 컴퓨터에 설치할 필요 없이, 이 페이지 안에서 바로 실행해볼 수 있습니다.
각 칸(셀) 왼쪽의 **▶ 재생 버튼**을 위에서 아래로 순서대로 누르기만 하면 됩니다.

> **안심하고 눌러보세요.** 이 노트북은 어떤 은행·증권 계좌에도 연결되어 있지 않습니다.
> 실제 돈은 절대 움직이지 않고, 화면에만 나오는 모의투자 결과입니다. 투자 자문이 아닌
> 연구·학습용 도구입니다.

## 1. 설치 (처음 한 번만 실행)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/rlawntjd19/fantastic-fortnight.git"
REPO_DIR = "fantastic-fortnight"

if os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_DIR)

!pip install -q -r requirements.txt
print("설치 완료! 아래로 내려가서 다음 칸을 실행하세요.")

## 2. (선택) 진짜 AI 설명 문구 쓰기

이 칸을 건너뛰어도 프로그램은 완전히 정상 동작합니다 — 다만 각 애널리스트의 설명이
`[offline-stub] ...` 같은 임시 문구로 나옵니다. 실제 자연어 설명을 보고 싶다면:

1. Colab 왼쪽 사이드바의 **열쇠 모양 아이콘(Secrets)** 클릭
2. **이름**: `ANTHROPIC_API_KEY`, **값**: 본인의 API 키 입력 후 저장, 좌측의 토글을 켜서
   이 노트북에 접근을 허용
3. 아래 칸 실행

(신호/신뢰도/레버리지 같은 숫자는 API 키 유무와 관계없이 항상 코드로 직접 계산되므로,
이 단계를 건너뛰어도 결과의 신뢰성에는 차이가 없습니다.)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API 키를 불러왔습니다.")
except Exception:
    print("건너뛰었습니다 — 위 안내대로 Secrets에 키를 등록한 뒤 다시 실행하면 됩니다. (선택사항)")

## 3. 분석 신호 생성해보기

아래 칸의 값을 원하는 대로 바꾼 뒤 실행하세요 (오른쪽의 입력창을 직접 클릭해서 수정 가능).
`leverage`를 아무리 크게 넣어도, 화면에는 안전 한도(기본 3배) 안으로 잘린 값이 나옵니다 —
의도된 동작입니다.

In [ ]:
symbol = "SK_HYNIX" #@param {type:"string"}
leverage = 5.0 #@param {type:"number"}
tranches = 2 #@param {type:"integer"}

!python -m trading_agent.cli signal "{symbol}" --leverage {leverage} --tranches {tranches}

## 4. (선택) 모의 계좌에 기록해보기

위에서 나온 계획을, 이 노트북 안에서만 존재하는 **가짜 모의투자 장부**에 기록해볼 수 있습니다.
실제 증권사·거래소에는 어떤 영향도 없습니다. 아래 `approve` 체크박스를 켠 뒤 실행하세요.

In [ ]:
approve = False #@param {type:"boolean"}

import dataclasses, os
from trading_agent.config import DEFAULT_CONFIG
from trading_agent.data.providers import SimulatedFeed
from trading_agent.engine.orchestrator import TradingCycle
from trading_agent.engine.paper_broker import PaperBroker
from trading_agent.engine.risk_controls import DailyCircuitBreaker
from trading_agent.llm.client import build_llm_client

# ANTHROPIC_API_KEY를 방금 등록했더라도 항상 최신 값을 반영하도록 새로 읽어옵니다.
config = dataclasses.replace(DEFAULT_CONFIG, anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY"))

llm = build_llm_client(config)
broker = PaperBroker(cash_equity=config.starting_paper_equity)
breaker = DailyCircuitBreaker(
    starting_equity=config.starting_paper_equity,
    limit_pct=config.risk.daily_loss_circuit_breaker_pct,
)
cycle = TradingCycle(config, llm, SimulatedFeed(), requested_leverage=leverage, requested_tranches=tranches)
artifacts = cycle.run_cycle(symbol, account_equity=broker.equity({}), circuit_breaker=breaker)

plan = artifacts.decision.trade_plan
print(f"{plan.action.value.upper()} {plan.symbol} @ {plan.entry_price:.2f} | "
      f"target {plan.target_price:.2f} | stop {plan.stop_loss_price:.2f}")
print(f"status={artifacts.decision.status}  leverage={artifacts.decision.risk_verdict.adjusted_leverage}x")

if approve and artifacts.decision.status == "pending_approval":
    broker.execute(artifacts.decision, human_approved=True)
    print("\n✅ 모의 계좌에 기록했습니다 (실제 자산과는 무관합니다).")
else:
    print("\n☐ 기록하지 않았습니다. 기록하려면 위 approve 체크박스를 켜고 이 칸을 다시 실행하세요.")

---
연구·학습용 도구입니다. 투자 자문이 아니며, 실제 매매·자산 운용에 대한 책임을 지지 않습니다.
더 자세한 옵션과 설계 배경은 저장소의 `USAGE.md` / `README.md`를 참고하세요.